In [1]:
library(Seurat)
library(scDiffCom)
library(data.table)

Attaching SeuratObject



In [2]:
sessionInfo()

R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: CentOS Linux 7 (Core)

Matrix products: default
BLAS/LAPACK: /work/project/ladcol_011/conda_path/miniconda3/envs/pyepianeu/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=de_DE.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=de_DE.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=de_DE.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=de_DE.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] data.table_1.17.8  scDiffCom_1.1.1    SeuratObject_4.1.4 Seurat_4.4.0      

loaded via a namespace (and not attached):
  [1] deldir_2.0-4           pbapply_1.7-4          gridExtra_2.3         
  [4]

In [3]:
input_dir <- "/work/project/ladcol_011/MariaWF/community-paper/src/data_preprocessing/Lasry/2.filtering/outs/"


counts <- fread(paste0(input_dir,"counts_lognorm.csv.gz"), header = TRUE)
counts <- as.data.frame(counts)
rownames(counts) <- counts$gene_symbol
counts <- counts[,-1]
print(str(counts))



'data.frame':	15770 obs. of  46702 variables:
 $ X2020.09.15.AML0024.CATCAAGGTTAGCGGA           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CATCAAGTCCGAGAAG           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CATCCACAGGGACCAT           : num  0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACAGAGCAAGA           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACAGTTCCATG           : num  0 0.993 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACGTAGAATAC           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACGTTCTCCCA           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACTCCGAACGC           : num  0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCAACTCTAGTCAG           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCACAAGACAGTCG           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCACACAATTGCCA           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020.09.15.AML0024.CCTCACACAGAACTAA           : num  0 0 0 0 0 0 0 0 0 0 ...
 $ X2020

In [4]:
anno_cells <- read.table(paste0(input_dir,"anno_cells_norm.txt")
                         ,sep = "\t"
                         # ,row.names = 1
                         ,header = TRUE
                         )
print(str(anno_cells))

'data.frame':	46702 obs. of  92 variables:
 $ sample_ID                    : chr  "AML-0024" "AML-0024" "AML-0024" "AML-0024" ...
 $ cell                         : chr  "2020-09-15-AML0024:CATCAAGGTTAGCGGA" "2020-09-15-AML0024:CATCAAGTCCGAGAAG" "2020-09-15-AML0024:CATCCACAGGGACCAT" "2020-09-15-AML0024:CCTCAACAGAGCAAGA" ...
 $ UMAP_1                       : num  -0.731 -2.2 -2.867 -1.666 -0.972 ...
 $ UMAP_2                       : num  -15.8 -16.7 -16.1 -16.1 -17.5 ...
 $ orig.ident                   : chr  "2020-09-15-AML0024" "2020-09-15-AML0024" "2020-09-15-AML0024" "2020-09-15-AML0024" ...
 $ samples                      : chr  "AML0024" "AML0024" "AML0024" "AML0024" ...
 $ Broad_cell_identity          : chr  "CD14+ monocyte" "CD14+ monocyte" "CD16+ monocyte" "CD14+ monocyte" ...
 $ Cell_type_identity           : chr  "CD14+ IFN+" "CD14+" "CD16+" "CD14+ IFN+" ...
 $ clusters_res.2               : int  7 7 7 7 80 7 7 7 7 7 ...
 $ CNV_pos                      : chr  "CNV+" "CNV+" "CN

In [5]:
counts_sparse <- Matrix::Matrix(as.matrix(counts), sparse = TRUE)

In [6]:
typeof(counts)

[1] "list"

In [7]:
typeof(counts_sparse)

[1] "S4"

In [8]:
colnames(counts_sparse) <- anno_cells$cell_ID

In [9]:
rownames(anno_cells) <- anno_cells$cell_ID

In [10]:
identical(colnames(counts_sparse), anno_cells$cell_ID)

[1] TRUE

In [11]:
counts_sparse[10:20, 1:10]

  [[ suppressing 10 column names ‘2020.09.15.AML0024.CATCAAGGTTAGCGGA’, ‘2020.09.15.AML0024.CATCAAGTCCGAGAAG’, ‘2020.09.15.AML0024.CATCCACAGGGACCAT’ ... ]]



11 x 10 sparse Matrix of class "dgCMatrix"
                                                                         
HES4       .        .        1.381679 .        .        .        .       
ISG15      2.448997 3.271507 3.020353 3.292314 1.732481 2.155702 2.773216
AGRN       .        .        .        .        .        .        .       
C1orf159   .        .        .        .        .        .        .       
TNFRSF18   .        .        .        .        .        .        .       
TNFRSF4    .        .        .        .        .        .        .       
SDF4       1.418543 .        .        .        .        .        .       
B3GALT6    .        .        .        .        .        .        .       
C1QTNF12   .        .        .        .        .        .        .       
AL162741.1 .        .        .        .        .        .        .       
UBE2J2     .        .        .        .        1.481063 .        .       
                                       
HES4       0.5299939 .       

In [23]:
obj <- CreateSeuratObject(counts = counts_sparse, meta.data = anno_cells)

In [24]:
obj[["RNA"]] <- CreateAssayObject(data = counts_sparse)

In [25]:
obj[["RNA"]]

Assay data with 15770 features for 46702 cells
First 10 features:
 AL627309.5, LINC01409, LINC01128, LINC00115, FAM41C, AL645608.2, NOC2L,
KLHL17, PLEKHN1, HES4 

In [26]:
DefaultAssay(obj) <- "RNA"

In [27]:
library(future)
plan(sequential)

In [28]:
tail(anno_cells$health_status)

[1] "healthy" "healthy" "healthy" "healthy" "healthy" "healthy"

In [29]:
class(obj@assays$RNA@data)

[1] "dgCMatrix"
attr(,"package")
[1] "Matrix"

In [30]:
class(obj@assays$RNA@counts)

[1] "matrix" "array"

In [36]:
scdiffcom_object <- run_interaction_analysis(
  seurat_object = obj,
  LRI_species = "human",
  seurat_celltype_id = "cell_type",
  seurat_condition_id = list(
    column_name = "health_status",
    cond1_name = "AML",
    cond2_name = "healthy"
  )
)

Extracting data from assay 'RNA' and slot 'data' (assuming normalized log1p-transformed data).

Converting normalized data from log1p-transformed to non-log1p-transformed.

Input data: 15770 genes, 46702 cells and 8 cell-types.

Input ligand-receptor database: 4785 human interactions.
Number of LRIs that match to genes present in the dataset: 1577.

Type of analysis to be performed: differential analysis between AML and healthy cells.

Total number of potential cell-cell interactions (CCIs): 100928 (8 * 8 * 1577).

Performing permutation analysis (1000 iterations by batches of 1000) on 8834 potential CCIs.

Performing batch 1 of 1.

Filtering and cleaning 'raw' CCIs.

Returning 3895 detected CCIs.

Performing over-representation analysis on the categories: LRI, LIGAND_COMPLEX, RECEPTOR_COMPLEX, ER_CELLTYPES, EMITTER_CELLTYPE, RECEIVER_CELLTYPE, GO_TERMS, KEGG_PWS.

Successfully returning final scDiffCom object.



In [44]:
CCI_detected <- GetTableCCI(scdiffcom_object, type = "detected", simplified = TRUE)

In [46]:
head(CCI_detected)

CCI,ER_CELLTYPES,LRI,NCELLS_EMITTER_AML,NCELLS_EMITTER_healthy,NCELLS_RECEIVER_AML,NCELLS_RECEIVER_healthy,CCI_SCORE_AML,CCI_SCORE_healthy,P_VALUE_AML,BH_P_VALUE_AML,P_VALUE_healthy,BH_P_VALUE_healthy,LOGFC,P_VALUE_DE,BH_P_VALUE_DE,IS_CCI_DETECTED_AML,IS_CCI_DETECTED_healthy,REGULATION
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<chr>
B_B_ADAM17:RHBDF2,B_B,ADAM17:RHBDF2,1515,3250,1515,3250,0.7995339,0.4986653,0,0,0.022,0.05625123,-0.47209387,0.000,0.00000000,TRUE,FALSE,DOWN
B_B_ADAM28:ITGA4,B_B,ADAM28:ITGA4,1515,3250,1515,3250,2.3712315,1.5392505,0,0,0.000,0.00000000,-0.43211385,0.000,0.00000000,TRUE,TRUE,DOWN
B_B_COPA:CD74,B_B,COPA:CD74,1515,3250,1515,3250,7.4996463,4.8507338,0,0,0.000,0.00000000,-0.43572587,0.000,0.00000000,TRUE,TRUE,DOWN
B_B_FAM3C:CLEC2D,B_B,FAM3C:CLEC2D,1515,3250,1515,3250,1.1911882,1.7526951,0,0,0.000,0.00000000,0.38620333,0.000,0.00000000,TRUE,TRUE,FLAT
B_B_FAM3C:HLA-C,B_B,FAM3C:HLA-C,1515,3250,1515,3250,2.9072357,3.2462576,0,0,0.000,0.00000000,0.11030012,0.008,0.01199864,TRUE,TRUE,FLAT
B_B_FAM3C:LAMP1,B_B,FAM3C:LAMP1,1515,3250,1515,3250,0.7537149,0.7708282,0,0,0.000,0.00000000,0.02245134,0.699,0.74254040,TRUE,TRUE,FLAT


In [47]:
ORA_results <- GetTableORA(scdiffcom_object, categories = "all", simplified = TRUE)